# 06 — Generator Selection

Manual, validation-only selection of one eligible fine-tuned and one eligible from-scratch generator. There is no approval script, signature, Git gate, or hidden automatic winner.

## Load benchmark results and registry

In [ ]:
from pathlib import Path
import json
import sys
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
import csv
from notebooks.utility.generator_benchmark import (load_protocol, load_registry, practical_equivalence,
    rank_generator_family, save_selected_generators, validate_selected_generators)
protocol = load_protocol(ROOT)
registry = load_registry(ROOT)
metrics_path = ROOT / protocol['outputs']['metrics']
benchmark_rows = list(csv.DictReader(metrics_path.open())) if metrics_path.is_file() else []
paired_path = ROOT / protocol['outputs']['paired_differences']
paired_rows = list(csv.DictReader(paired_path.open())) if paired_path.is_file() else []
benchmark_rows[:3] if benchmark_rows else 'Not yet evaluated'

## Eligible family candidates and technical gates

In [ ]:
filtered_rows = [row for row in benchmark_rows if row.get('condition') == 'FILTERED']
finetuned_ranking = rank_generator_family(filtered_rows, 'finetuned', protocol['eligibility_gates']) if filtered_rows else []
fromscratch_ranking = rank_generator_family(filtered_rows, 'from_scratch', protocol['eligibility_gates']) if filtered_rows else []
fine_tuned_candidates = [row for row in finetuned_ranking if row['eligible'] and row.get('role') == 'primary_candidate']
from_scratch_candidates = [row for row in fromscratch_ranking if row['eligible'] and row.get('role') == 'primary_candidate']
sampling_ablations = [row for row in finetuned_ranking + fromscratch_ranking if row.get('role') == 'sampling_ablation']
descriptive_baselines = [row for row in finetuned_ranking + fromscratch_ranking if row.get('role') == 'descriptive_baseline']
excluded_candidates = [row for row in finetuned_ranking + fromscratch_ranking if not row['eligible']]
{'fine-tuned candidates': fine_tuned_candidates, 'from-scratch candidates': from_scratch_candidates,
 'sampling ablations': sampling_ablations, 'descriptive baselines': descriptive_baselines,
 'excluded candidates and reasons': [(row['generator_id'], row['exclusion_reasons']) for row in excluded_candidates]}

## Full-pool KID, balanced PRDC point estimates, repeated-subsampling stability, provenance, memorization and efficiency

In [ ]:
display_columns = ['generator_id', 'family_rank', 'raddino_kid', 'raddino_kid_stability_low', 'raddino_kid_stability_high',
                   'raddino_coverage', 'raddino_precision', 'raddino_fid', 'inception_kid',
                   'raddino_kid_std', 'technical_validity_rate', 'filter_acceptance_rate',  # descriptive only
                   'provenance_manifest_valid', 'lineage_complete', 'training_corpus_manifest',
                   'train_memorization_rate', 'synthetic_duplicate_rate', 'generation_seconds_per_image', 'efficiency_status']
[[{column: row.get(column) for column in display_columns} for row in ranking] for ranking in (finetuned_ranking, fromscratch_ranking)]

## Practical equivalence and manual decision

In [ ]:
PROPOSED_FINETUNED_GENERATOR = next((row['generator_id'] for row in finetuned_ranking if row['eligible']), None)
PROPOSED_FROM_SCRATCH_GENERATOR = next((row['generator_id'] for row in fromscratch_ranking if row['eligible']), None)
SELECTED_FINETUNED_GENERATOR = PROPOSED_FINETUNED_GENERATOR
SELECTED_FROM_SCRATCH_GENERATOR = PROPOSED_FROM_SCRATCH_GENERATOR
paired_by_family = {row['family']: row for row in paired_rows}
equivalence_notes = {family: practical_equivalence(row, protocol) for family, row in paired_by_family.items()}
SELECTION_NOTES = 'Automatic ordering uses full-reference estimates. Practical similarity requires a paired stability interval containing zero and the preregistered mean-difference margin.'
{'proposed': (PROPOSED_FINETUNED_GENERATOR, PROPOSED_FROM_SCRATCH_GENERATOR), 'practical_equivalence': equivalence_notes}

## Validate and save the simple selection file

In [ ]:
SAVE_SELECTION = False
if SAVE_SELECTION:
    selected = validate_selected_generators(SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR,
                                            registry, benchmark_rows, protocol['synthetic_pool_target'])
    output = save_selected_generators(ROOT, selected['finetuned'], selected['from_scratch'], benchmark_rows,
                                      notes=SELECTION_NOTES,
                                      manual_override=(SELECTED_FINETUNED_GENERATOR != PROPOSED_FINETUNED_GENERATOR or SELECTED_FROM_SCRATCH_GENERATOR != PROPOSED_FROM_SCRATCH_GENERATOR))
    print(output)
else:
    print('Selection not saved. Set both IDs after reviewing real benchmark results, then set SAVE_SELECTION=True.')